# 04 - Forecasting Evaluation Audit & WAPE Investigation

**Purpose:** determine whether WAPE = 0.4001 (LightGBM winner) is genuinely poor
forecasting, an artifact of the metric/aggregation, a consequence of sparse
intermittent demand, or acceptable for the downstream pricing objective.

**Method:** reproduce the *exact* Phase-3 validation evaluation using the
persisted winner (`artifacts/phase3/model.joblib`, `feature_columns.json`) and
`data/processed/demand_matrix_weekly.csv.gz` - the same matrix, feature build,
chronological split, imputation and design matrix as notebook 03. No re-fitting,
no Optuna, no test-split access.

Diagnosis follows: implementation audit -> granularity -> zero/intermittent
demand -> error segmentation -> metric comparison -> validation audit ->
baselines -> root-cause classification -> business interpretation.

## 1. Setup

In [1]:
import sys, json
from pathlib import Path

_cwd = Path.cwd()
def _find_root(path: Path) -> Path:
    while path != path.parent:
        if (path / "src").is_dir() and (path / "data").is_dir():
            return path
        path = path.parent
    return path

ROOT = _find_root(_cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib

from src.utils.logging_setup import get_logger
from src.utils.metrics import describe_metrics, wape, mae, rmse, asymmetric_loss
from src.data.target_engineering import load_demand_matrix
from src.features.feature_engineering import build_phase3_feature_matrix
from src.data.temporal_split import temporal_split, assert_chronological_order
from src.data.imputation import impute_splits
from src.models.data_prep import prepare_model_matrix, design_matrix_rows
from src.models.forecasting import naive_combined_predictions

LOGGER = get_logger("notebook_04")

model_path = ROOT / "artifacts" / "phase3" / "model.joblib"
feat_path = ROOT / "artifacts" / "phase3" / "feature_columns.json"
winner = joblib.load(model_path)
FEATURE_COLUMNS = json.loads(feat_path.read_text(encoding="utf-8"))
print(f"Loaded winner estimator: {type(winner).__name__}")
print(f"Feature schema: {len(FEATURE_COLUMNS)} columns")

Loaded winner estimator: LGBMRegressor
Feature schema: 86 columns


## 2. Reproduce the validation evaluation exactly as notebook 03

Same matrix -> Phase-3 features -> chronological 70/15/15 -> train-only
imputation -> global-category design matrix -> active-column schema ->
winner.predict -> expm1. This must reproduce **WAPE = 0.4001**.

In [2]:
matrix = load_demand_matrix(ROOT / "data" / "processed" / "demand_matrix_weekly.csv.gz")
features_full = build_phase3_feature_matrix(matrix)
del matrix

splits = temporal_split(features_full, "week", ratios=(0.70, 0.15, 0.15))
del features_full
assert_chronological_order(splits, "week")

NUMERIC_IMPUTE = ["avg_price", "avg_freight", "price_prev", "price_chg_1w", "freight_prev"]
splits = impute_splits(splits, categorical_cols=["product_category_name"], numeric_cols=NUMERIC_IMPUTE)

global_categories = sorted(
    pd.concat([f["product_category_name"] for f in splits.values()]).astype(str).unique()
)
prepared = {name: prepare_model_matrix(frame, categories=global_categories)
            for name, frame in splits.items()}

train_X, train_y, train_w = design_matrix_rows(prepared["train"])
val_X, val_y, val_w = design_matrix_rows(prepared["val"])

train_X = train_X[FEATURE_COLUMNS].copy()
val_X = val_X[FEATURE_COLUMNS].copy()

pred_log = winner.predict(val_X)
pred = pd.Series(np.expm1(pred_log), index=val_X.index, name="pred_demand")

actual = prepared["val"]["order_demand"].astype("float64").reset_index(drop=True)
pred = pred.reset_index(drop=True)
weeks = prepared["val"]["week"].reset_index(drop=True)
pids = prepared["val"]["product_id"].astype(str).reset_index(drop=True)

eval_df = pd.DataFrame({
    "product_id": pids,
    "week": weeks.astype(str),
    "actual": actual,
    "pred": pred,
    "err": actual - pred,
    "abs_err": (actual - pred).abs(),
})
del train_X, val_X, train_y, train_w, val_y, val_w, pred_log, prepared
print(f"Validation population: {len(eval_df):,} product-week rows, "
      f"{eval_df['product_id'].nunique():,} products, {eval_df['week'].nunique()} weeks")

rep = describe_metrics(eval_df["actual"].to_numpy(), eval_df["pred"].to_numpy())
print("Reproduced metrics (expect wape=0.4001):")
for k, v in rep.items():
    print(f"  {k:<22} {v:.4f}")
assert abs(rep["wape"] - 0.4001) < 0.001, "Did NOT reproduce the Phase-3 validation WAPE."

2026-09-22 12:35:16 | INFO    | src.data.target_engineering | load_demand_matrix: loaded 3469274 rows x 6 cols.


2026-09-22 12:35:16 | INFO    | src.features.feature_engineering | build_phase3_feature_matrix: starting (3469274 rows).


2026-09-22 12:35:24 | INFO    | src.features.feature_engineering | add_demand_drivers: price_prev, price_chg_1w, freight_prev, category_share_prev added.


2026-09-22 12:35:28 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 4) added.


2026-09-22 12:35:41 | INFO    | src.features.feature_engineering | add_rolling_features: windows (4,) aggs ('mean', 'std') added (shifted-by-1, past-only): ['rolling_demand_mean_4w', 'rolling_demand_std_4w']


...

2026-09-22 12:35:58 | INFO    | src.data.imputation | impute_splits: val remaining NaNs=0


2026-09-22 12:36:05 | INFO    | src.models.data_prep | prepare_model_matrix: 2428492 rows x 86 design cols (incl. 73 one-hot categories).


2026-09-22 12:36:06 | INFO    | src.models.data_prep | prepare_model_matrix: 520391 rows x 86 design cols (incl. 73 one-hot categories).


2026-09-22 12:36:08 | INFO    | src.models.data_prep | prepare_model_matrix: 520391 rows x 86 design cols (incl. 73 one-hot categories).


Validation population: 520,391 product-week rows, 32,729 products, 17 weeks
Reproduced metrics (expect wape=0.4001):
  wape                   0.4001
  mae                    0.0235
  rmse                   0.2725
  asymmetric_loss_2_1    0.0381


## 3. WAPE implementation audit (exact formulas)

From `src/utils/metrics.py`:

- `wape(a, p) = sum |a - p| / sum a` — **global** (numerator- and denominator-
  summed first), NOT `mean(|a-p|/a)`.
- If `sum a == 0` the function returns `0.0` (perfect agreement guard).
- `mae(a, p) = mean |a - p|`; `rmse = sqrt(mean((a-p)^2))`.
- `asymmetric_loss(a, p)` = `mean(w * |a-p|)` with `w = 2` when `a > p`
  (under-forecast) and `w = 1` otherwise.
- `predictions` are the raw `expm1` of the log-domain model output; WAPE is
  evaluated on the **business scale** (units demanded).
- **No filtering, no sample weights, no per-product averaging** feed WAPE.

So the exact reported quantity is:

    WAPE = (sum over val product-week rows |order_demand - pred_demand|)
         / (sum over val product-week rows order_demand)

equivalent to the **total-demand-weighted** average of product-level WAPEs
(weights = product demand over the validation window).

## 4. Error budget: zero-demand rows vs positive-demand rows

In [3]:
zero_mask = eval_df["actual"] == 0
pos_mask = ~zero_mask
tot_demand = eval_df["actual"].sum()

print(f"Zero-demand val rows: {zero_mask.sum():,} / {len(eval_df):,} "
      f"({zero_mask.mean()*100:.1f}%)")
print(f"Positive-demand val rows: {pos_mask.sum():,} "
      f"({pos_mask.mean()*100:.1f}%); total demand units: {tot_demand:,.0f}")

budget = {
    "zero_rows_abs_err": float(eval_df.loc[zero_mask, "abs_err"].sum()),
    "pos_rows_abs_err": float(eval_df.loc[pos_mask, "abs_err"].sum()),
    "wape_contribution_zero_rows": float(eval_df.loc[zero_mask, "abs_err"].sum() / tot_demand),
    "wape_contribution_pos_rows": float(eval_df.loc[pos_mask, "abs_err"].sum() / tot_demand),
    "wape_on_pos_rows_only": wape(eval_df.loc[pos_mask, "actual"], eval_df.loc[pos_mask, "pred"]),
    "mean_pred_on_zero_rows": float(eval_df.loc[zero_mask, "pred"].mean()),
    "mae_on_zero_rows": float(eval_df.loc[zero_mask, "abs_err"].mean()),
    "mae_on_pos_rows": float(eval_df.loc[pos_mask, "abs_err"].mean()),
}
for k, v in budget.items():
    print(f"  {k:<30} {v:.4f}")

# Under/over decomposition
under = eval_df["err"] > 0
over = eval_df["err"] < 0
print(f"\nOn ALL rows: under-forecast weeks {under.mean()*100:.1f}%, "
      f"over-forecast weeks {over.mean()*100:.1f}%")
under_units = eval_df.loc[under, "err"].sum()
over_units = -eval_df.loc[over, "err"].sum()
print(f"Under-forecast deficit: {under_units:,.0f} units "
      f"({under_units/tot_demand*100:.1f}% of total demand)")
print(f"Over-forecast surplus: {over_units:,.0f} units "
      f"({over_units/tot_demand*100:.1f}% of total demand)")
print(f"Cost-weighted error (2:1) = {(2*under_units + 1*over_units)/tot_demand:.4f} "
      f"(i.e. WAPE under 2:1 stockout costs)")

Zero-demand val rows: 499,990 / 520,391 (96.1%)
Positive-demand val rows: 20,401 (3.9%); total demand units: 30,578
  zero_rows_abs_err              855.5250
  pos_rows_abs_err               11377.5221
  wape_contribution_zero_rows    0.0280
  wape_contribution_pos_rows     0.3721
  wape_on_pos_rows_only          0.3721
  mean_pred_on_zero_rows         0.0017
  mae_on_zero_rows               0.0017
  mae_on_pos_rows                0.5577

On ALL rows: under-forecast weeks 1.2%, over-forecast weeks 98.8%
Under-forecast deficit: 7,585 units (24.8% of total demand)
Over-forecast surplus: 4,648 units (15.2% of total demand)
Cost-weighted error (2:1) = 0.6481 (i.e. WAPE under 2:1 stockout costs)


## 5. Error segmentation: demand volume, maturity, pattern, direction

In [4]:
# ---- Product metadata from TRAIN (for segmentation only; no leakage) ----
tr = splits["train"][["product_id", "order_demand", "week"]].copy()
tr["product_id"] = tr["product_id"].astype(str)
grp = tr.groupby("product_id", as_index=False)["order_demand"].agg(
    train_demand="sum", train_pos_weeks=lambda s: int((s > 0).sum()), train_weeks="count",
)
grp["train_zero_frac"] = 1 - grp["train_pos_weeks"] / grp["train_weeks"]
nonzero = tr[tr["order_demand"] > 0].groupby("product_id")["order_demand"]
grp["train_mean_nonzero"] = grp["product_id"].map(nonzero.mean()).fillna(0.0)
grp["train_cv"] = (grp["product_id"].map(nonzero.std())
                   / grp["product_id"].map(nonzero.mean())).fillna(0.0)

meta_map = grp.set_index("product_id")

eval_seg = eval_df.copy()
for col in ["train_demand", "train_pos_weeks", "train_weeks",
            "train_zero_frac", "train_mean_nonzero", "train_cv"]:
    eval_seg[col] = eval_seg["product_id"].map(meta_map[col])

# Buckets: avg weekly train demand = train_demand / train_weeks
eval_seg["avg_wkly"] = eval_seg["train_demand"] / eval_seg["train_weeks"].clip(lower=1)
def vol_bucket(x):
    if pd.isna(x):  return "T0 no train history"
    if x < 1:      return "T1 light <1/wk"
    if x < 2:      return "T2 low 1-2/wk"
    if x < 5:      return "T3 medium 2-5/wk"
    if x < 10:     return "T4 high 5-10/wk"
    return "T5 very high >10/wk"
eval_seg["volume_bucket"] = eval_seg["avg_wkly"].map(vol_bucket)

def mature_bucket(w):
    if pd.isna(w):  return "M0 no train history"
    if w <= 1:  return "M1 launch (<=1 sale wk)"
    if w <= 5:  return "M2 short (2-5)"
    if w <= 12: return "M3 developing (6-12)"
    if w <= 30: return "M4 mature (13-30)"
    return "M5 long-history (31-75)"
eval_seg["maturity_bucket"] = eval_seg["train_pos_weeks"].map(mature_bucket)

def pattern_bucket(zf, cv):
    if pd.isna(zf):  return "P0 no train history"
    if zf <= 0.10: return "P1 continuous/stable"
    if zf <= 0.40: return "P2 mostly-steady intermittent"
    if zf <= 0.80: return "P3 intermittent (40-80% zeros)"
    return "P4 sparse (>80% zeros)"
eval_seg["pattern_bucket"] = [pattern_bucket(z, c) for z, c in
                              zip(eval_seg["train_zero_frac"], eval_seg["train_cv"])]

def seg_table(frame, seg_col, weight_note="(global WAPE per segment = demand-weighted)"):
    rows = []
    for name, g in frame.groupby(seg_col):
        if g["actual"].sum() <= 0:
            continue
        rows.append({
            "segment": name,
            "rows": len(g),
            "products": g["product_id"].nunique(),
            "demand_units": float(g["actual"].sum()),
            "wape": wape(g["actual"], g["pred"]),
            "mae": mae(g["actual"], g["pred"]),
            "under%": float((g["err"] > 0).mean() * 100),
            "bias_units": float((g["pred"] - g["actual"]).mean()),
            "bias_pct": float((g["pred"].sum() - g["actual"].sum()) / g["actual"].sum() * 100),
        })
    out = pd.DataFrame(rows).set_index("segment")
    out = out.sort_values("demand_units", ascending=False)
    print(out.round(4).to_string())
    return out

print("=" * 88)
print("BY TRAIN-VOLUME BUCKET")
print("=" * 88)
vol_table = seg_table(eval_seg, "volume_bucket")

print("\n" + "=" * 88)
print("BY MATURITY (train weeks with positive demand)")
print("=" * 88)
mat_table = seg_table(eval_seg, "maturity_bucket")

print("\n" + "=" * 88)
print("BY DEMAND PATTERN (train zero fraction + CV)")
print("=" * 88)
pat_table = seg_table(eval_seg, "pattern_bucket")

BY TRAIN-VOLUME BUCKET


                    rows  products  demand_units    wape     mae   under%  bias_units  bias_pct
segment                                                                                        
T1 light <1/wk    519786     32691       28731.0  0.3903  0.0216   1.1688     -0.0046   -8.2758
T3 medium 2-5/wk     159        10         908.0  0.5384  3.0745  43.3962     -1.7477  -30.6042
T2 low 1-2/wk        430        27         874.0  0.5673  1.1531  31.8605     -0.6057  -29.8007
T4 high 5-10/wk       16         1          65.0  0.5281  2.1456  56.2500     -1.2731  -31.3382

BY MATURITY (train weeks with positive demand)


                           rows  products  demand_units    wape     mae   under%  bias_units  bias_pct
segment                                                                                               
M1 launch (<=1 sale wk)  405683     25517       16211.0  0.3563  0.0142   0.8731     -0.0025   -6.3341
M2 short (2-5)            93726      5892        6421.0  0.3960  0.0271   1.3774     -0.0045   -6.5804
M3 developing (6-12)      16258      1023        3998.0  0.4684  0.1152   5.1236     -0.0348  -14.1628
M4 mature (13-30)          4376       275        2968.0  0.5292  0.3589  11.4488     -0.1621  -23.8951
M5 long-history (31-75)     348        22         980.0  0.4805  1.3531  35.3448     -0.6082  -21.5976

BY DEMAND PATTERN (train zero fraction + CV)


                                  rows  products  demand_units    wape     mae   under%  bias_units  bias_pct
segment                                                                                                      
P4 sparse (>80% zeros)          517129     32524       27374.0  0.3854  0.0204   1.1241     -0.0041   -7.8376
P3 intermittent (40-80% zeros)    3214       202        3072.0  0.5274  0.5041  14.2190     -0.2382  -24.9210
P2 mostly-steady intermittent       48         3         132.0  0.4738  1.3030  41.6667     -0.5292  -19.2454


## 6. Category-level performance

In [5]:
cat_map = splits["val"][["product_id", "product_category_name"]].drop_duplicates("product_id")
cat_map["product_id"] = cat_map["product_id"].astype(str)
cat = eval_seg[["product_id", "actual", "pred", "err"]].merge(
    cat_map, on="product_id", how="left")
cat["product_category_name"] = cat["product_category_name"].astype(object).fillna("UNKNOWN")

cat_rows = []
for name, g in cat.groupby("product_category_name"):
    if g["actual"].sum() <= 0:
        continue
    cat_rows.append({
        "category": name,
        "rows": len(g),
        "demand_units": float(g["actual"].sum()),
        "wape": wape(g["actual"], g["pred"]),
        "bias_pct": float((g["pred"].sum() - g["actual"].sum()) / g["actual"].sum() * 100),
    })
cat_table = pd.DataFrame(cat_rows).sort_values("demand_units", ascending=False)

top = cat_table.head(15)
print("TOP-15 CATEGORIES BY VALIDATION DEMAND")
print(top.round(4).to_string(index=False))

worst = cat_table.nlargest(8, "wape")
print("\nWORST 8 CATEGORIES BY WAPE")
print(worst.round(4).to_string(index=False))

concentration = cat_table.set_index("category")["demand_units"]
share = float(cat_table.loc[cat_table["demand_units"] >= cat_table["demand_units"].quantile(0.2), "demand_units"].sum())
print(f"\nTop-20% demand categories carry {share/cat_table['demand_units'].sum()*100:.1f}% of validation demand")

err_by_cat = (cat["actual"] - cat["pred"]).abs().groupby(cat["product_category_name"]).sum()
top_err_cat = err_by_cat.sort_values(ascending=False).head(5)
print("Categories contributing most absolute dollar-error (units):")
for c, v in top_err_cat.items():
    print(f"  {c:<40} {v:,.0f}")

TOP-15 CATEGORIES BY VALIDATION DEMAND
              category   rows  demand_units   wape  bias_pct
               unknown 501419       22046.0 0.3765   -8.0446
informatica_acessorios   1497        1352.0 0.5213  -24.1660
       cama_mesa_banho   1990         891.0 0.4570   -9.0280
          beleza_saude   1417         840.0 0.4789  -12.0405
    relogios_presentes   1162         713.0 0.4978  -21.0779
    ferramentas_jardim    318         668.0 0.5068  -14.3916
         esporte_lazer   2133         634.0 0.4594   -8.6526
           eletronicos    798         500.0 0.4610  -17.0133
             telefonia    908         451.0 0.4516   -8.4676
            cool_stuff    445         264.0 0.4558  -13.1847
 utilidades_domesticas    968         258.0 0.4449   -0.1656
      moveis_decoracao    925         250.0 0.4291   -8.0687
            perfumaria    669         231.0 0.4648    1.2765
            automotivo    893         207.0 0.3681   -4.1907
     moveis_escritorio    301         133.0 0.

## 7. Product-level WAPE distribution (aggregation check)

In [6]:
prod = eval_seg.groupby("product_id").apply(
    lambda g: pd.Series({
        "demand": float(g["actual"].sum()),
        "wape": wape(g["actual"], g["pred"]),
        "mae": mae(g["actual"], g["pred"]),
        "n_weeks": len(g),
    }), include_groups=False).reset_index()

prod = prod[prod["demand"] > 0]
prod = prod[~prod["wape"].isna()]
print(f"Products with positive val demand: {len(prod):,}")

prod_w_dist = {
    "global_wape (=demand-weighted product WAPE)": wape(eval_df["actual"], eval_df["pred"]),
    "unweighted mean of product WAPEs": float(prod["wape"].mean()),
    "demand-weighted mean of product WAPEs": float((prod["wape"] * prod["demand"]).sum() / prod["demand"].sum()),
    "median product WAPE": float(prod["wape"].median()),
    "p25 / p75 product WAPE": (float(prod["wape"].quantile(0.25)), float(prod["wape"].quantile(0.75))),
    "share products WAPE<0.25": float((prod["wape"] < 0.25).mean()),
    "share products WAPE<0.50": float((prod["wape"] < 0.50).mean()),
    "share products WAPE 0.5-1.0": float(((prod["wape"] >= 0.50) & (prod["wape"] < 1.0)).mean()),
    "share products WAPE>1.0": float((prod["wape"] >= 1.0).mean()),
}
for k, v in prod_w_dist.items():
    if isinstance(v, tuple):
        print(f"  {k:<45} {v[0]:.4f} / {v[1]:.4f}")
    else:
        print(f"  {k:<45} {v:.4f}")

corr = prod[["demand", "wape"]].corr(method="spearman").iloc[0, 1]
print(f"\nSpearman(product val demand, product WAPE) = {corr:.3f}")

vol = eval_seg["volume_bucket"]
print("\nWAPE by bucket under UNWEIGHTED mean vs global (aggregation sensitivity):")
for b in eval_seg["volume_bucket"].unique():
    g = eval_seg[eval_seg["volume_bucket"] == b]
    if g["actual"].sum() <= 0:
        continue
    print(f"  {b:<20} unweighted mean {g.groupby('product_id').apply(lambda x: wape(x['actual'], x['pred']), include_groups=False).mean():.3f}   "
          f"global {wape(g['actual'], g['pred']):.3f}")

2026-09-22 12:36:56 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


2026-09-22 12:36:56 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


2026-09-22 12:36:56 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


2026-09-22 12:36:56 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


2026-09-22 12:36:56 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


...

2026-09-22 12:38:51 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


  T1 light <1/wk       unweighted mean 0.089   global 0.390


  T2 low 1-2/wk        unweighted mean 0.555   global 0.567
2026-09-22 12:38:51 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


  T3 medium 2-5/wk     unweighted mean 0.469   global 0.538


  T4 high 5-10/wk      unweighted mean 0.528   global 0.528


## 8. Metric comparison & additional diagnostics

In [7]:
a = eval_df["actual"].to_numpy(float)
p = eval_df["pred"].to_numpy(float)
log_a = np.log1p(a); log_p = np.log1p(p)

smape_denom = (np.abs(a) + np.abs(p))
eps = smape_denom == 0
smape_denom[eps] = 1e-12
smape = float(np.mean(2 * np.abs(a - p) / smape_denom))
rmsle = float(np.sqrt(np.mean((log_a - log_p) ** 2)))
mean_signed_err = float(np.mean(p - a))
bias_norm = float(np.mean((p - a) / np.maximum(a, 1e-12)))

# MASE-style: naive persistence MAE on the same rows (naive remainder)
y_tn, y_pn = naive_combined_predictions(splits["train"], splits["val"])
mase_naive_mae = mae(y_tn, y_pn)

metrics_row = {
    "wape": wape(a, p),
    "mae": mae(a, p),
    "rmse": rmse(a, p),
    "asymmetric_loss_2_1": asymmetric_loss(a, p),
    "naive_mae_val": mase_naive_mae,
    "mae_ratio_vs_naive": mae(a, p) / mase_naive_mae,
    "smape": smape,
    "rmsle": rmsle,
    "mean_signed_error_units": mean_signed_err,
    "bias_percent_of_demand": float((p.sum() - a.sum()) / a.sum() * 100),
}
mr = pd.Series(metrics_row)
print(mr.round(4).to_string())

print("\nDirectional concentration (demand>0 rows only):")
ep = eval_df.loc[eval_df["actual"] > 0]
under_p = ep["err"] > 0
print(f"  positive-demand weeks under-forecast: {under_p.mean()*100:.1f}%")
print(f"  mean signed error on positive rows: {float((ep['pred'] - ep['actual']).mean()):.4f} units")
print(f"  under-weighted cost share: "
      f"{100*2*float(ep['err'][under_p].sum()) / (2*float(ep['err'][under_p].sum()) - float(ep['err'][~under_p].sum())):.1f}% of 2:1 cost")

wape                       0.4001
mae                        0.0235
rmse                       0.2725
asymmetric_loss_2_1        0.0381
naive_mae_val              0.0875
mae_ratio_vs_naive         0.2687
smape                      1.9335
rmsle                      0.0566
mean_signed_error_units   -0.0056
bias_percent_of_demand    -9.6031

Directional concentration (demand>0 rows only):
  positive-demand weeks under-forecast: 29.6%
  mean signed error on positive rows: -0.1856 units
  under-weighted cost share: 80.0% of 2:1 cost


## 9. Diagnostic baselines on the SAME population

No more tuning - just forecast-edges. Every baseline is evaluated on the exact
same 520,391 validation rows with the same expm1-unit-scale contract.

In [8]:
# ---- Baseline predictors ----
def rolling_mean_predict(train, evalset, window=4):
    cols = ["product_id", "week", "order_demand"]
    combined = pd.concat([train[cols], evalset[cols]], ignore_index=True)
    combined = combined.sort_values(["product_id", "week"], kind="mergesort")
    combined["prev"] = combined.groupby("product_id", observed=True)["order_demand"].shift(1)
    combined["ma"] = combined.groupby("product_id", observed=True)["prev"].transform(
        lambda s: s.rolling(window, min_periods=1).mean())
    combo = combined.groupby(["product_id", "week"], observed=True)["ma"].first().reset_index()
    merged = evalset[["product_id", "week", "order_demand"]].merge(
        combo, on=["product_id", "week"], how="left")
    mask = merged["ma"].notna()
    return merged.loc[mask, "order_demand"].to_numpy(float), merged.loc[mask, "ma"].to_numpy(float)

def monthly_mean_predict(train, evalset):
    t = train.copy(); t["month"] = t["week"].dt.month
    v = evalset.copy(); v["month"] = v["week"].dt.month
    stats = t.groupby(["product_id", "month"], observed=True)["order_demand"].mean()
    prod_mean = t.groupby("product_id", observed=True)["order_demand"].mean()
    m = v.merge(stats.rename("mm"), left_on=["product_id", "month"], right_index=True, how="left")
    m["mm"] = m["mm"].fillna(m["product_id"].map(prod_mean)).fillna(0.0)
    return m["order_demand"].to_numpy(float), m["mm"].to_numpy(float)

def hybrid_predict(train, evalset):
    """Persistence for active products, 0 for never-seen-before products."""
    cols = ["product_id", "week", "order_demand"]
    combined = pd.concat([train[cols], evalset[cols]], ignore_index=True)
    combined = combined.sort_values(["product_id", "week"], kind="mergesort")
    combined["pred"] = combined.groupby("product_id", observed=True)["order_demand"].shift(1)
    combined["pred"] = combined["pred"].fillna(0.0)
    combo = combined.groupby(["product_id", "week"], observed=True)["pred"].first().reset_index()
    merged = evalset[["product_id", "week", "order_demand"]].merge(combo, on=["product_id", "week"], how="left")
    return merged["order_demand"].to_numpy(float), merged["pred"].to_numpy(float)

baselines = {}
baselines["naive_persistence"] = naive_combined_predictions(splits["train"], splits["val"])
baselines["moving_average_4w"] = rolling_mean_predict(splits["train"], splits["val"], window=4)
baselines["monthly_mean"] = monthly_mean_predict(splits["train"], splits["val"])
baselines["persist_or_zero"] = hybrid_predict(splits["train"], splits["val"])
baselines["winner_lgbm"] = (eval_df["actual"].to_numpy(), eval_df["pred"].to_numpy())

rows = {"model": [], "rows": [], "wape": [], "mae": [], "asym21": [], "bias_pct": []}
for name, (ya, yp) in baselines.items():
    rows["model"].append(name)
    rows["rows"].append(len(ya))
    rows["wape"].append(wape(ya, yp))
    rows["mae"].append(mae(ya, yp))
    rows["asym21"].append(asymmetric_loss(ya, yp))
    rows["bias_pct"].append((yp.sum() - ya.sum()) / ya.sum() * 100)
base_table = pd.DataFrame(rows).set_index("model").sort_values("wape")
print(base_table.round(4).to_string())

# Relative lift of the winner vs best baseline
best_bl = base_table.drop(index="winner_lgbm").sort_values("wape").iloc[0]
lift = (best_bl["wape"] - base_table.loc["winner_lgbm", "wape"]) / best_bl["wape"] * 100
print(f"\nWinner WAPE is {lift:.1f}% better than the best baseline "
      f"({base_table.drop(index='winner_lgbm').sort_values('wape').index[0]}).")

# Insample naive MAE for MASE scale (train only - standard MASE denominator)
ins_naive = splits["train"][["product_id", "week", "order_demand"]].sort_values(
    ["product_id", "week"], kind="mergesort")
ins_naive = ins_naive.assign(pred=ins_naive.groupby("product_id", observed=True)["order_demand"].shift(1))
ins_naive = ins_naive[ins_naive["pred"].notna()]
mase = mae(eval_df["actual"], eval_df["pred"]) / mae(ins_naive["order_demand"], ins_naive["pred"])
print(f"\nMASE (val MAE / train 1-step naive MAE) = {mase:.4f}")

                     rows    wape     mae  asym21  bias_pct
model                                                      
winner_lgbm        520391  0.4001  0.0235  0.0381   -9.6031
monthly_mean       520391  1.3036  0.0766  0.1341  -65.2945
moving_average_4w  520391  1.4721  0.0865  0.1303   -1.9671
persist_or_zero    520391  1.4910  0.0876  0.1316   -0.7260
naive_persistence  526937  1.4923  0.0875  0.1314   -0.8192

Winner WAPE is 69.3% better than the best baseline (monthly_mean).



MASE (val MAE / train 1-step naive MAE) = 0.6179


## 10. Feature & weekly stability diagnostics

In [9]:
# Weekly WAPE progression across the 17 val weeks
wk = eval_df.groupby("week").apply(
    lambda g: pd.Series({"wape": wape(g["actual"], g["pred"]),
                         "demand": float(g["actual"].sum()),
                         "rows": len(g)}), include_groups=False)
print("Weekly WAPE progression (validation):")
print(wk.round(4).to_string())

# Model evidence used: gain importances
booster = getattr(winner, "booster_", winner)
if hasattr(booster, "feature_importance"):
    imp = pd.Series(
        booster.feature_importance(importance_type="gain"),
        index=booster.feature_name(),
    ).sort_values(ascending=False)
    print("\nTop-12 features by GAIN importance (winner LightGBM):")
    print(imp.head(12).round(0).to_string())

# Zero-rows share and model behavior on them by volume bucket
zr = eval_seg[eval_seg["actual"] == 0]
print("\nZero-demand predictions by volume bucket (should ideally be near 0):")
zsum = zr.groupby("volume_bucket").apply(
    lambda g: pd.Series({"rows": len(g), "mean_pred": float(g["pred"].mean()),
                         "p90_pred": float(g["pred"].quantile(0.90)),
                         "share_pred>0.5": float((g["pred"] > 0.5).mean())}),
    include_groups=False)
print(zsum.round(4).to_string())

Weekly WAPE progression (validation):
                         wape  demand     rows
week                                          
2018-01-29/2018-02-04  0.4248  1507.0  26183.0
2018-02-05/2018-02-11  0.4173  1744.0  32729.0
2018-02-12/2018-02-18  0.4075  1860.0  32729.0
2018-02-19/2018-02-25  0.4182  1998.0  32729.0
2018-02-26/2018-03-04  0.4101  2145.0  32729.0
2018-03-05/2018-03-11  0.3885  1859.0  32729.0
2018-03-12/2018-03-18  0.3887  1802.0  32729.0
2018-03-19/2018-03-25  0.3946  1992.0  32729.0
2018-03-26/2018-04-01  0.3837  1681.0  32729.0
2018-04-02/2018-04-08  0.3773  1798.0  32729.0
2018-04-09/2018-04-15  0.3853  1802.0  32729.0
2018-04-16/2018-04-22  0.3950  1928.0  32729.0
2018-04-23/2018-04-29  0.3881  1912.0  32729.0
2018-04-30/2018-05-06  0.4122  2024.0  32729.0
2018-05-07/2018-05-13  0.4127  2287.0  32729.0
2018-05-14/2018-05-20  0.3944  2130.0  32729.0
2018-05-21/2018-05-27  0.4023   109.0   3273.0

Top-12 features by GAIN importance (winner LightGBM):
cat_unknown   


Zero-demand predictions by volume bucket (should ideally be near 0):


                      rows  mean_pred  p90_pred  share_pred>0.5
volume_bucket                                                  
T1 light <1/wk    499777.0     0.0017    0.0016          0.0000
T2 low 1-2/wk        177.0     0.0471    0.0962          0.0226
T3 medium 2-5/wk      35.0     0.0657    0.1100          0.0571
T4 high 5-10/wk        1.0     0.0907    0.0907          0.0000


## 11. Validation-methodology audit table

Verified by code inspection (not by trusting docs):

| Item | Implementation | Verdict |
|---|---|---|
| Chronological split | `temporal_split` sort by week, `<=` boundaries, `assert_chronological_order` guard | OK |
| Feature leak-free build | all features `shift(k)`/shift-by-1-rolling; equivalence proof in notebook 03 | OK |
| Train-only imputation | `SimpleImputer(median)` fit on train only; unknown label for unseen categories | OK |
| Expanding-window CV | 5 folds, min_train 24w, horizon 4w, validation always after train | OK |
| Tuning population | 120 mature products (train only); hyperparams chosen on mean-fold WAPE | OK* |
| Early stopping | last 8 weeks of train as eval set - inside training boundary only | OK |
| Sample weighting | `1 + log1p(demand)` applied to log-domain LGBM/XGB fits | OK |
| LGBM constant-feature guard | active-column schema exact across fit/predict | OK |
| Test split | never fitted, never scored | OK |

*Caveat: Optuna optimizes on mature products but the final model is scored on
*all* products in val; tuning and reporting populations differ (noted, not a
leak). CV WAPE range 0.49-0.66 (mature) vs full-population val 0.40 - a
population effect, not leakage.

## 12. Root-cause classification summary

Quantified contributor shares are printed above. Interpretation tables and the
full report are produced by the analyst from these numbers.

In [10]:
print("Audit diagnostics complete. Validation WAPE reproduced:", round(rep["wape"], 4))

Audit diagnostics complete. Validation WAPE reproduced: 0.4001
